In [54]:
import re
import pandas as pd
from rapidfuzz.distance import JaroWinkler
import itertools

def std_name(name):
    prefixes = r'^(?:(?:Adv|Dr|Mr|Mrs|Ms|Shri|Smt|Prof)\.?\s+)+'
    name = re.sub(prefixes, "", name, flags=re.IGNORECASE)

    lineage_field = r'\s+(?:S/o|D/o|W/o|C/o|H/o|@)\.?\s+.*$'
    name = re.sub(lineage_field, "", name, flags=re.IGNORECASE)

    name = name.split(",", 1)[0]
    name = name.split("(", 1)[0]
    name = name.replace(".", " ")

    words = name.split()
    words = [word.lower() for word in words]
    return " ".join(words)

def names_agree(a, b, threshold=0.80):
    words_a = a.split()
    words_b = b.split()
    if len(words_a) <= len(words_b):
        short, long = words_a, words_b
    else:
        short, long = words_b, words_a

    def words_match(w1, w2):
        if len(w1) == 1 or len(w2) == 1:
            return w1[0] == w2[0]
        return JaroWinkler.similarity(w1, w2) >= threshold

    for perm in itertools.permutations(long, len(short)):
        if all(words_match(s, l) for s, l in zip(short, perm)):
            return True
    return False


def match_candidates(
    election_df,
    winners_df,
    election_name_col="Candidate",
    winner_name_col="Winner Candidate",
):
    election_df = election_df.copy()
    winners_df = winners_df.copy()

    election_df = election_df.set_index("Sno.")

    election_df["_std_name"] = election_df[election_name_col].apply(std_name)
    winners_df["_std_name"] = winners_df[winner_name_col].apply(std_name)

    matches = []

    # Outer loop: winners
    for i, winner_row in winners_df.iterrows():
        name1 = winner_row["_std_name"]

        if not name1:
            continue

        best_match = None
        best_score = 0

        # Inner loop: election candidates
        for j, election_row in election_df.iterrows():
            name2 = election_row["_std_name"]

            if not name2:
                continue

            if names_agree(name1, name2):
                score = JaroWinkler.similarity(name1, name2)

                if score > best_score:
                    best_score = score
                    best_match = {
                        "winner_candidate": winner_row[winner_name_col],
                        "winner_of_constituency": winner_row["Constituency"],
                        "election_candidate": election_row[election_name_col],
                        "similarity": score,
                        "party": winner_row.get("Winner Party", None),
                        "winner_index": i,
                        "election_index": j
                    }

        if best_match is not None:
            matches.append(best_match)

    return pd.DataFrame(matches)

In [55]:
df = pd.read_csv("loksabha_winners_2014_2019_2024_normalized.csv")

In [56]:
andhra_14_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2014)]
andhra_14 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2014_imputed.parquet")
matching_andhra_14 = match_candidates(andhra_14, andhra_14_winners)
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,147
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,235
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,16
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,70
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,186
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,283
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,210
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,184
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,88
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,310


**Manual Checking**

In [57]:
# Check Sno. for Y. S. Avinash Reddy
for _, row in andhra_14.iterrows():
    if names_agree(std_name(row["Candidate"]), "y s avinash reddy"):
        print(row["Candidate"], row["Sno."])

Avinash Reddy Yeduguri Sandinti 18
Shanthi Reddy 260
Yeluri Srilatha 328


In [58]:
matching_andhra_14.loc[matching_andhra_14["winner_candidate"] == "Y. S. Avinash Reddy", "election_candidate"] = "Avinash Reddy Yeduguri Sandinti"
matching_andhra_14.loc[matching_andhra_14["winner_candidate"] == "Y. S. Avinash Reddy", "election_index"] = 18
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,147
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,235
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,16
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,70
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,186
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,283
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,210
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,184
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,88
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,310


In [59]:
andhra_19_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2019)]
andhra_19 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2019_imputed.parquet")
matching_andhra_19 = match_candidates(andhra_19, andhra_19_winners)
matching_andhra_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Goddeti Madhavi,Araku (ST),Goddeti Madhavi,1.000000,YSRCP,91,97
1,Kinjarapu Ram Mohan Naidu,Srikakulam,Rammohannaidu Kinjarapu,0.721678,TDP,92,246
2,Bellana Chandra Sekhar,Vizianagaram,Bellana Chandra Sekhar,1.000000,YSRCP,93,32
3,M. V. V. Satyanarayana,Visakhapatnam,M V V Satyanarayana,1.000000,YSRCP,94,163
4,Beesetti Venkata Satyavathi,Anakapalle,Dr Beesetti Venkata Satyavathi,1.000000,YSRCP,95,78
5,Vanga Geetha,Kakinada,Vanga Geetha Viswanatham,0.900000,YSRCP,96,296
6,Chinta Anuradha,Amalapuram (SC),Chinta Anuradha,1.000000,YSRCP,97,55
7,Margani Bharat,Rajahmundry,Margani Bharat Ram,0.955556,YSRCP,98,178
8,Raghu Rama Krishna Raju,Narasapuram,Kanumuru Raghu Rama Krishna Raju,0.761322,YSRCP,99,141
9,Kotagiri Sridhar,Eluru,K T Sreedhar,0.622685,YSRCP,100,132


In [60]:
andhra_24_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2024)]
andhra_24 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2024_imputed.parquet")
matching_andhra_24 = match_candidates(andhra_24, andhra_24_winners)
matching_andhra_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Gumma Thanuja Rani,Araku (ST),Gumma Thanuja Rani,1.000000,YSRCP,182,176
1,K. Ram Mohan Naidu,Srikakulam,Kinjarapu Rammohan Naidu,0.751103,TDP,183,223
2,Appala Naidu Kalisetti,Vizianagaram,Appalanaidu Kalisetti,0.962338,TDP,184,33
3,C. M. Ramesh,Anakapalle,C M Ramesh,1.000000,BJP,186,96
4,Tangella Uday Srinivas,Kakinada,Tangella Uday Srinivas,1.000000,Janasena Party,187,392
5,Ganti Harish Madhur,Amalapuram (SC),G M Harish,0.657018,TDP,188,150
6,Daggubati Purandeswari,Rajahmundry,Daggubati Purandheshwari,0.983333,BJP,189,116
7,Bhupathi Srinivasa Varma,Narasapuram,Bhupathi Raju Srinivasa Varma,0.907184,BJP,190,71
8,Putta Mahesh Kumar,Eluru,Putta Mahesh Kumar,1.000000,TDP,191,337
9,Vallabhaneni Balashowry,Machilipatnam,Balashowry Vallabhaneni,0.620083,Janasena Party,192,53


In [61]:
bihar_24_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2024)]
bihar_24 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2024_imputed.parquet")
matching_bihar_24 = match_candidates(bihar_24, bihar_24_winners)
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,445
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,129
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,312
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,202
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,105
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,58
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,356
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,113
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,295
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,235


**Manual Checking**

In [62]:
# Check Sno. for Lalan Singh
for _, row in bihar_24.iterrows():
    if names_agree(std_name(row["Candidate"]), "lalan singh"):
        print(row["Candidate"], row["Sno."])

Lal Badsah Singh 199
Rajiv Ranjan Singh Alias Lalan Singh 335


In [63]:
matching_bihar_24.loc[matching_bihar_24["winner_candidate"] == "Lalan Singh", "election_candidate"] = "Rajiv Ranjan Singh Alias Lalan Singh"
matching_bihar_24.loc[matching_bihar_24["winner_candidate"] == "Lalan Singh", "election_index"] = 335
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,445
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,129
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,312
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,202
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,105
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,58
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,356
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,113
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,295
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,235


In [64]:
for i in list(bihar_24["Candidate"]):
    if names_agree(std_name(i), "abhay kushwaha"):
        print(i)

A K Singh


In [65]:
bihar_19_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2019)]
bihar_19 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2019_imputed.parquet")
matching_bihar_19 = match_candidates(bihar_19, bihar_19_winners)
matching_bihar_19


,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Baidyanath Prasad Mahto,Valmiki Nagar,Baidyanath Prasad Mahto,1.000000,JD(U),116,81
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jaiswal,1.000000,BJP,117,156
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,118,352
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,119,411
4,Sunil Kumar Pintu,Sitamarhi,Sunil Kumar,0.929412,JD(U),120,543
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,121,75
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),122,429
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),123,137
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,124,332
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,125,154


In [66]:
bihar_14_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2014)]
bihar_14 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2014_imputed.parquet")
matching_bihar_14 = match_candidates(bihar_14, bihar_14_winners)
matching_bihar_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Satish Chandra Dubey,Valmiki Nagar,Satish Dubey,0.920000,BJP,25,486
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jayaswal,0.943663,BJP,26,135
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,27,382
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,28,431
4,Ram Kumar Sharma,Sitamarhi,Ram Kumar Sharma,1.000000,Rashtriya Lok Samta Party,29,424
5,Hukmdev Narayan Yadav,Madhubani,Hukum Deo Narayan Yadav,0.900121,BJP,30,166
6,Birendra Chaudhary,Jhanjharpur,Bijendra Chaudhary,0.938998,BJP,31,76
7,Ranjeet Ranjan,Supaul,Ranjeet Ranjan,1.000000,INC,32,451
8,Taslimuddin,Araria,Taslim Uddin,0.983333,RJD,33,566
9,Tariq Anwar,Katihar,Md Tariq Anwar,0.837662,NCP,35,271


In [67]:
gujarat_14_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2014)]
gujarat_14 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2014_imputed.parquet")
matching_gujarat_14 = match_candidates(gujarat_14, gujarat_14_winners)
matching_gujarat_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhmashi,0.694264,BJP,65,44
1,Haribhai Chaudhary,Banaskantha,Harilal Ranchodbhai Chauhan,0.844227,BJP,66,82
2,Liladhar Vaghela,Patan,Vaghela Liladharbhai Khodaji,0.711310,BJP,67,314
3,Jayshreeben Patel,Mahesana,Patel Jayshreeben Kanubhai,0.747360,BJP,68,194
4,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,69,236
5,L. K. Advani,Gandhinagar,Lal Krishna Advani,0.746667,BJP,70,118
6,Paresh Rawal,Ahmedabad East,Paresh Dahyalal Rawal,0.880952,BJP,71,173
7,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai Premjibhai,0.667582,BJP,72,67
8,Devajibhai Fatepara,Surendranagar,Fatepara Devajibhai Govindbhai,0.719883,BJP,73,69
9,Mohan Kundariya,Rajkot,Mohanbhai Kalyanjibhai Kundariya,0.641667,BJP,74,149


In [68]:
gujarat_19_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2019)]
gujarat_19 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2019_imputed.parquet")
matching_gujarat_19 = match_candidates(gujarat_19, gujarat_19_winners)
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,70
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,207
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,300
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,270
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,10
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,223
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,99
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,169
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,160
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,259


In [69]:
for _, row in gujarat_19.iterrows():
    if names_agree(std_name(row["Candidate"]), std_name("Rajesh Chudasama")):
        print(row["Candidate"], row["Sno."])

Chudasama Rajeshbhai Naranbhai 73
Rakeshbhai 257


In [70]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "Rajesh Chudasama", "election_candidate"] = "Chudasama Rajeshbhai Naranbhai"
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "Rajesh Chudasama", "election_index"] = 73
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,70
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,207
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,300
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,270
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,10
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,223
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,99
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,169
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,160
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,259


In [71]:
for _, row in gujarat_19.iterrows():
    if names_agree(std_name(row["Candidate"]), std_name("C. R. Patil")):
        print(row["Candidate"], row["Sno."])

Chandansinh 45
Chandraben 96
Patil Chandrakant 239
Rakeshbhai 257
Reshma Patel 280
Rita Captain 281


In [72]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "C. R. Patil", "election_candidate"] = "Patil Chandrakant"
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "C. R. Patil", "election_index"] = 239
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,70
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,207
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,300
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,270
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,10
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,223
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,99
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,169
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,160
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,259


In [73]:
gujarat_24_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2024)]
gujarat_24 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2024_imputed.parquet")
matching_gujarat_24 = match_candidates(gujarat_24, gujarat_24_winners)
matching_gujarat_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,247,59
1,Geni Thakor,Banaskantha,Geniben Nagaji Thakor,0.610390,INC,248,89
2,Bharatsinhji Dabhi,Patan,Dabhi Bharatsinhji Shankarji,0.732804,BJP,249,64
3,Haribhai Patel,Mahesana,Haribhai Patel,1.000000,BJP,250,95
4,Shobhanaben Baraiya,Sabarkantha,Shobhanaben Mahendrasinh Baraiya,0.887171,BJP,251,230
5,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,252,9
6,Hasmukh Patel,Ahmedabad East,Hasmukhbhai Patel,0.937557,BJP,253,98
7,Dinesh Makwana,Ahmedabad West,Dineshbhai Makwana,0.898413,BJP,254,80
8,Chandubhai Shihora,Surendranagar,Chandubhai Chhaganbhai Shihora,0.886667,BJP,255,51
9,Parshottam Rupala,Rajkot,Parshottambhai Rupala,0.926611,BJP,256,185


In [74]:
def match_repeat_candidates(
    year1_df,
    year2_df,
    name_col="Candidate",
    threshold=0.85
):
    results = []

    a = year1_df.copy()
    b = year2_df.copy()

    a["_std_name"] = a[name_col].fillna("").apply(std_name)
    b["_std_name"] = b[name_col].fillna("").apply(std_name)

    for i, row1 in a.iterrows():
        name1 = row1["_std_name"]
        age1 = row1['Age']
        if not name1:
            continue

        candidates = []

        for j, row2 in b.iterrows():
            name2 = row2["_std_name"]
            age2 = row2['Age']
            if not name2 or not names_agree(name1, name2) or not (4<=age2-age1<=5):
                continue

            score = JaroWinkler.similarity(name1, name2)

            if score < threshold:
                continue

            candidates.append({
                "candidate_2014": row1[name_col],
                "candidate_2019": row2[name_col],
                "similarity": score,
                "index_prev": row1['Sno.'],
                "index_cur": row2['Sno.']
            })

        if candidates:
            best = max(
                candidates,
                key=lambda x: x["similarity"]
            )
            results.append(best)

    return pd.DataFrame(results)

In [75]:
def flag_recontesters(df_prev, df_curr, name_col="Candidate", threshold=0.70):
    # 1. Get the raw matches using your existing function
    matches = match_repeat_candidates(df_prev, df_curr, name_col=name_col, threshold=threshold)
    
    # Safety check: if no matches exist at all
    if matches.empty:
        df_curr["Recontester"] = -1
        return df_curr

    # 2. Create the 1:1 mapping, resolving duplicates by highest similarity
    mapping = (
        matches
        .sort_values("similarity", ascending=False)
        .drop_duplicates("index_cur")
        .set_index("index_cur")["index_prev"]
    )
    
    # 3. Map Sno. of current year candidates to Sno. of previous year
    df_curr["Recontester"] = df_curr["Sno."].map(mapping).fillna(-1).astype(int)
    
    return df_curr

In [76]:
andhra_19 = flag_recontesters(andhra_14, andhra_19)
andhra_24 = flag_recontesters(andhra_19, andhra_24)
bihar_19 = flag_recontesters(bihar_14, bihar_19)
bihar_24 = flag_recontesters(bihar_19, bihar_24)
gujarat_19 = flag_recontesters(gujarat_14, gujarat_19)
gujarat_24 = flag_recontesters(gujarat_19, gujarat_24)

**Adding new columns to data**

In [77]:
def process_election_data(df, matching_df):
    df = df.copy()

    # Map constituency by election_index (Sno.)
    constituency_map = matching_df.set_index("election_index")["winner_of_constituency"]
    
    # Identify which candidates won using Sno.
    winning_snos = matching_df["election_index"].unique()
    
    # Mark winner status and map constituency using Sno.
    df["winner"] = df["Sno."].isin(winning_snos)
    df["constituency"] = df["Sno."].map(constituency_map).fillna("")

    return df

In [78]:
datasets = [
    (andhra_14, matching_andhra_14, "andhra_2014"),
    (andhra_19, matching_andhra_19, "andhra_2019"),
    (andhra_24, matching_andhra_24, "andhra_2024"),
    (bihar_14, matching_bihar_14, "bihar_2014"),
    (bihar_19, matching_bihar_19, "bihar_2019"),
    (bihar_24, matching_bihar_24, "bihar_2024"),
    (gujarat_14, matching_gujarat_14, "gujarat_2014"),
    (gujarat_19, matching_gujarat_19, "gujarat_2019"),
    (gujarat_24, matching_gujarat_24, "gujarat_2024"),
]

processed_dfs = {}
for df, matching_df, name in datasets:
    processed_dfs[name] = process_election_data(df, matching_df)

In [79]:
for name in processed_dfs:
    dataframe = processed_dfs[name]
    dataframe.to_parquet(f"cleaned_data/{name}.parquet", index=False)